# Q12.
```{admonition}
:class: note
Apply boosting, bagging, random forests, and BART to a data set of your choice. Be sure to fit the models on a training set and to evaluate their performance on a test set. How accurate are the results compared to simple methods like linear or logistic regression? Which of these approaches yields the best performance?

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.ensemble import GradientBoostingRegressor, BaggingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score

from ISLP.bart import BART

In [3]:
wage = pd.read_csv('../../../ALL CSV FILES - 2nd Edition/Wage.csv')

In [4]:
wage.sample(3)

,year,age,maritl,race,education,region,jobclass,health,health_ins,logwage,wage
1280,2009,59,3. Widowed,2. Black,2. HS Grad,2. Middle Atlantic,2. Information,1. <=Good,1. Yes,4.653213,104.921507
1004,2005,43,2. Married,1. White,2. HS Grad,2. Middle Atlantic,2. Information,1. <=Good,1. Yes,4.230449,68.748088
1027,2006,47,2. Married,1. White,4. College Grad,2. Middle Atlantic,1. Industrial,2. >=Very Good,2. No,4.462398,86.695155


In [5]:
wageD = pd.get_dummies(wage,dtype=int)

In [6]:
wageD.sample(3)

,year,age,logwage,wage,maritl_1. Never Married,maritl_2. Married,maritl_3. Widowed,maritl_4. Divorced,maritl_5. Separated,race_1. White,...,education_3. Some College,education_4. College Grad,education_5. Advanced Degree,region_2. Middle Atlantic,jobclass_1. Industrial,jobclass_2. Information,health_1. <=Good,health_2. >=Very Good,health_ins_1. Yes,health_ins_2. No
2701,2004,35,4.778151,118.884359,1,0,0,0,0,1,...,0,0,1,1,0,1,0,1,1,0
1576,2006,53,4.698970,109.833986,0,1,0,0,0,1,...,0,1,0,1,0,1,0,1,1,0
1056,2004,25,4.113943,61.187526,0,1,0,0,0,1,...,0,0,0,1,1,0,0,1,1,0


In [7]:
X_train, X_test, y_train, y_test = train_test_split(wageD.drop(columns=['wage','logwage']),wageD['logwage'],random_state=1728,shuffle=True)

In [8]:
models = [GradientBoostingRegressor(learning_rate=0.01,n_estimators=500),
          GradientBoostingRegressor(learning_rate=0.01,n_estimators=5000,max_depth=1),
          RandomForestRegressor(n_jobs=-1,n_estimators=500,max_features=X_train.shape[1],random_state=1728),
          RandomForestRegressor(n_jobs=-1,n_estimators=500,max_features='sqrt',random_state=1728),
          LinearRegression(),
          BART(random_state=1728,burnin=50,ndraw=200)
          ]

model_names = ['Boosting (3)','Boosting (1)','Bagging','Random Forest', 'Linear Regression','BART']

In [9]:
scores_dict = {}

for i, model in enumerate(models):
    model.fit(X_train,y_train)
    y_test_pred = model.predict(X_test)
    model_name = model_names[i]
    scores_dict[model_name] = {}
    scores_dict[model_name]['CV MSE'] = -np.mean(cross_val_score(model,X_train,y_train,n_jobs=-1,scoring='neg_mean_squared_error'))
    scores_dict[model_name]['Test MSE'] = mean_squared_error(y_test,y_test_pred)
    scores_dict[model_name]['Test R2'] = r2_score(y_test,y_test_pred)

In [15]:
pd.DataFrame(scores_dict).sort_values(axis=1,by='Test MSE')

,Boosting (1),Boosting (3),BART,Linear Regression,Random Forest,Bagging
CV MSE,0.078860,0.079979,0.078728,0.080304,0.094538,0.096669
Test MSE,0.070661,0.070677,0.070883,0.073034,0.081584,0.084995
Test R2,0.374230,0.374083,0.372259,0.353216,0.277490,0.247287
